In [2]:
import sys

print("Python:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python:
c:\Users\amrat\AppData\Local\Programs\Python\Python312\python.exe

Python version:
3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]


In [3]:
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "show", "scikit-learn"],
    capture_output=False
)

CompletedProcess(args=['c:\\Users\\amrat\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'show', 'scikit-learn'], returncode=1)

In [4]:
import sys
!{sys.executable} -m pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -----------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import sklearn

print("sklearn version:", sklearn.__version__)

sklearn version: 1.9.1


In [6]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
movie_features = pd.read_csv(
    "../data/processed/movie_features.csv"
)

movie_features.head()

,movieId,title,genres,tag,content
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Owned imdb top 250 Pixar Pixar time travel chi...,Adventure Animation Children Comedy Fantasy Ow...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Robin Williams time travel fantasy based on ch...,Adventure Children Fantasy Robin Williams time...
2,3,Grumpier Old Men (1995),Comedy|Romance,funny best friend duringcreditsstinger fishing...,Comedy Romance funny best friend duringcredits...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,based on novel or book chick flick divorce int...,Comedy Drama Romance based on novel or book ch...
4,5,Father of the Bride Part II (1995),Comedy,aging baby confidence contraception daughter g...,Comedy aging baby confidence contraception dau...


In [8]:
movie_features.shape

(62423, 5)

In [9]:
movie_features[["title", "content"]].head(10)

,title,content
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy Ow...
1,Jumanji (1995),Adventure Children Fantasy Robin Williams time...
2,Grumpier Old Men (1995),Comedy Romance funny best friend duringcredits...
3,Waiting to Exhale (1995),Comedy Drama Romance based on novel or book ch...
4,Father of the Bride Part II (1995),Comedy aging baby confidence contraception dau...
5,Heat (1995),Action Crime Thriller imdb top 250 great actin...
6,Sabrina (1995),Comedy Romance remake chauffeur fusion long is...
7,Tom and Huck (1995),Adventure Children based on a book Mark Twain ...
8,Sudden Death (1995),Action explosive hostage terrorist vice presid...
9,GoldenEye (1995),Action Adventure Thriller 007 Bond boys with t...


In [10]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000
)

In [11]:
tfidf_matrix = tfidf.fit_transform(
    movie_features["content"]
)

In [12]:
tfidf_matrix.shape

(62423, 35768)

In [14]:
movie_indices = pd.Series(
    movie_features.index,
    index=movie_features["title"]
).drop_duplicates()

movie_indices.head()

title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64

In [15]:
movie_indices["Toy Story (1995)"]

np.int64(0)

In [16]:
movie_idx = movie_indices["Toy Story (1995)"]

movie_vector = tfidf_matrix[movie_idx]

movie_vector.shape

(1, 35768)

In [17]:
from sklearn.metrics.pairwise import linear_kernel

In [18]:
movie_idx = movie_indices["Toy Story (1995)"]

similarity_scores = linear_kernel(
    tfidf_matrix[movie_idx],
    tfidf_matrix
).flatten()

similarity_scores.shape

(62423,)

In [19]:
similar_movie_indices = similarity_scores.argsort()[-11:][::-1]

similar_movie_indices

array([    0,  3021,  2264,  4780, 14813, 26560, 59767, 39485, 48035,
        6258,  8246])

In [20]:
movie_features.iloc[
    similar_movie_indices
][["movieId", "title", "genres"]]

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy
4780,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
14813,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX
26560,126405,The Adventures of André and Wally B. (1984),Animation
59767,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy
39485,157296,Finding Dory (2016),Adventure|Animation|Comedy
48035,175831,Lou (2017),Animation
6258,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy


In [21]:
def recommend_movies(movie_title, n=10):
    if movie_title not in movie_indices:
        return f"Movie '{movie_title}' not found."

    movie_idx = movie_indices[movie_title]

    similarity_scores = linear_kernel(
        tfidf_matrix[movie_idx],
        tfidf_matrix
    ).flatten()

    similar_indices = similarity_scores.argsort()[-(n + 1):][::-1]

    similar_indices = [
        idx for idx in similar_indices
        if idx != movie_idx
    ][:n]

    recommendations = movie_features.iloc[similar_indices][
        ["movieId", "title", "genres"]
    ].copy()

    recommendations["similarity_score"] = similarity_scores[similar_indices]

    return recommendations

In [22]:
recommend_movies("Toy Story (1995)")

,movieId,title,genres,similarity_score
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.929208
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.848912
4780,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.785346
14813,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.769655
26560,126405,The Adventures of André and Wally B. (1984),Animation,0.716924
59767,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,0.716768
39485,157296,Finding Dory (2016),Adventure|Animation|Comedy,0.716011
48035,175831,Lou (2017),Animation,0.713410
6258,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.711302
8246,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.705333


In [23]:
recommend_movies("Jumanji (1995)")

,movieId,title,genres,similarity_score
1638,1702,Flubber (1997),Children|Comedy|Fantasy,0.675188
8224,8939,"Final Cut, The (2004)",Sci-Fi|Thriller,0.642272
749,765,Jack (1996),Comedy|Drama,0.594261
495,500,Mrs. Doubtfire (1993),Comedy|Drama,0.571158
3352,3448,"Good Morning, Vietnam (1987)",Comedy|Drama|War,0.549846
24158,120805,Robin Williams: Weapons of Self Destruction (2...,Comedy,0.524246
13794,71429,World's Greatest Dad (2009),Comedy|Drama,0.502686
3391,3489,Hook (1991),Adventure|Comedy|Fantasy,0.488187
5420,5528,One Hour Photo (2002),Drama|Thriller,0.473789
11695,53974,License to Wed (2007),Comedy|Romance,0.444538


In [24]:
recommend_movies("This Movie Does Not Exist")

"Movie 'This Movie Does Not Exist' not found."

In [25]:
recommend_movies("Toy Story (1995)", n=5)

,movieId,title,genres,similarity_score
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.929208
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.848912
4780,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.785346
14813,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.769655
26560,126405,The Adventures of André and Wally B. (1984),Animation,0.716924


In [26]:
toy_story_recs = recommend_movies("Toy Story (1995)", n=10)

toy_story_recs

,movieId,title,genres,similarity_score
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.929208
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.848912
4780,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.785346
14813,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.769655
26560,126405,The Adventures of André and Wally B. (1984),Animation,0.716924
59767,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,0.716768
39485,157296,Finding Dory (2016),Adventure|Animation|Comedy,0.716011
48035,175831,Lou (2017),Animation,0.713410
6258,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.711302
8246,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.705333


In [27]:
toy_story_recs[["title", "similarity_score"]]

,title,similarity_score
3021,Toy Story 2 (1999),0.929208
2264,"Bug's Life, A (1998)",0.848912
4780,"Monsters, Inc. (2001)",0.785346
14813,Toy Story 3 (2010),0.769655
26560,The Adventures of André and Wally B. (1984),0.716924
59767,Toy Story 4 (2019),0.716768
39485,Finding Dory (2016),0.716011
48035,Lou (2017),0.713410
6258,Finding Nemo (2003),0.711302
8246,"Incredibles, The (2004)",0.705333


In [29]:
test_movies = [
    "Toy Story (1995)",
    "Jumanji (1995)",
    "The Matrix (1999)",
    "Titanic (1997)"
]

for movie in test_movies:
    print(f"\nRecommendations for: {movie}")
    
    result = recommend_movies(movie, n=5)
    
    if isinstance(result, str):
        print(result)
    else:
        display(
            result[["title", "genres", "similarity_score"]]
        )


Recommendations for: Toy Story (1995)


,title,genres,similarity_score
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.929208
2264,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.848912
4780,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.785346
14813,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.769655
26560,The Adventures of André and Wally B. (1984),Animation,0.716924



Recommendations for: Jumanji (1995)


,title,genres,similarity_score
1638,Flubber (1997),Children|Comedy|Fantasy,0.675188
8224,"Final Cut, The (2004)",Sci-Fi|Thriller,0.642272
749,Jack (1996),Comedy|Drama,0.594261
495,Mrs. Doubtfire (1993),Comedy|Drama,0.571158
3352,"Good Morning, Vietnam (1987)",Comedy|Drama|War,0.549846



Recommendations for: The Matrix (1999)
Movie 'The Matrix (1999)' not found.

Recommendations for: Titanic (1997)


,title,genres,similarity_score
9442,"Aviator, The (2004)",Drama,0.494080
145,"Basketball Diaries, The (1995)",Drama,0.462598
20651,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama,0.453980
5877,Catch Me If You Can (2002),Crime|Drama,0.453780
11238,Blood Diamond (2006),Action|Adventure|Crime|Drama|Thriller|War,0.435929


In [30]:
(movie_features["tag"].str.strip() == "").sum()

np.int64(0)

In [31]:
(
    (movie_features["tag"].str.strip() == "").sum()
    / len(movie_features)
) * 100

np.float64(0.0)

In [33]:
import joblib

In [34]:
joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

['../models/tfidf_vectorizer.pkl']

In [35]:
import os

os.path.exists("../models/tfidf_vectorizer.pkl")

True

In [36]:
loaded_tfidf = joblib.load(
    "../models/tfidf_vectorizer.pkl"
)

print(type(loaded_tfidf))

<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [37]:
movie_features["content_length"] = (
    movie_features["content"]
    .fillna("")
    .str.split()
    .str.len()
)

movie_features["content_length"].describe()

count    62423.000000
mean        29.693991
std        144.762910
min          1.000000
25%          3.000000
50%          6.000000
75%         15.000000
max      10581.000000
Name: content_length, dtype: float64

In [38]:
movie_features[
    movie_features["content_length"] <= 2
][["movieId", "title", "genres", "tag", "content"]].head(20)

,movieId,title,genres,tag,content
83,84,Last Summer in the Hamptons (1995),Comedy|Drama,NaN,Comedy Drama
89,90,The Journey of August King (1995),Drama,NaN,Drama
106,108,Catwalk (1996),Documentary,NaN,Documentary
137,139,Target (1995),Action|Drama,NaN,Action Drama
140,142,Shadows (Cienie) (1988),Drama,NaN,Drama
190,192,The Show (1995),Documentary,NaN,Documentary
198,200,"Tie That Binds, The (1995)",Thriller,NaN,Thriller
281,284,New York Cop (Nyû Yôku no koppu) (1993),Action|Crime,NaN,Action Crime
284,287,Nina Takes a Lover (1994),Comedy|Romance,NaN,Comedy Romance
306,310,Rent-a-Kid (1995),Comedy,NaN,Comedy


In [39]:
(movie_features["content_length"] > 5).sum()

np.int64(33118)

In [40]:
(
    (movie_features["content_length"] > 5).sum()
    / len(movie_features)
) * 100

np.float64(53.054162728481494)

In [41]:
movie_features["weighted_content"] = (
    movie_features["genres"].str.replace("|", " ", regex=False) + " "
    + movie_features["genres"].str.replace("|", " ", regex=False) + " "
    + movie_features["tag"].fillna("")
)

In [42]:
weighted_tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000
)

weighted_tfidf_matrix = weighted_tfidf.fit_transform(
    movie_features["weighted_content"]
)

print(weighted_tfidf_matrix.shape)

(62423, 35768)


In [43]:
def recommend_movies_weighted(movie_title, n=10):
    if movie_title not in movie_indices:
        return f"Movie '{movie_title}' not found."

    movie_idx = movie_indices[movie_title]

    similarity_scores = linear_kernel(
        weighted_tfidf_matrix[movie_idx],
        weighted_tfidf_matrix
    ).flatten()

    similar_indices = similarity_scores.argsort()[-(n + 1):][::-1]

    similar_indices = [
        idx for idx in similar_indices
        if idx != movie_idx
    ][:n]

    recommendations = movie_features.iloc[similar_indices][
        ["movieId", "title", "genres"]
    ].copy()

    recommendations["similarity_score"] = similarity_scores[
        similar_indices
    ]

    return recommendations

In [44]:
recommend_movies_weighted("Toy Story (1995)", n=10)

,movieId,title,genres,similarity_score
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.930007
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.851211
4780,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.786317
14813,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.770890
59767,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,0.730895
39485,157296,Finding Dory (2016),Adventure|Animation|Comedy,0.718785
6258,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.712220
8246,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.706227
19870,103141,Monsters University (2013),Adventure|Animation|Comedy,0.699954
48035,175831,Lou (2017),Animation,0.673460


In [45]:
recommend_movies_weighted("The Matrix (1999)", n=10)

"Movie 'The Matrix (1999)' not found."

In [46]:
original_recs = recommend_movies("Toy Story (1995)", n=10)

weighted_recs = recommend_movies_weighted("Toy Story (1995)", n=10)

print("ORIGINAL MODEL")
display(
    original_recs[["title", "genres", "similarity_score"]]
)

print("\nWEIGHTED MODEL")
display(
    weighted_recs[["title", "genres", "similarity_score"]]
)

ORIGINAL MODEL


,title,genres,similarity_score
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.929208
2264,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.848912
4780,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.785346
14813,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.769655
26560,The Adventures of André and Wally B. (1984),Animation,0.716924
59767,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,0.716768
39485,Finding Dory (2016),Adventure|Animation|Comedy,0.716011
48035,Lou (2017),Animation,0.713410
6258,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.711302
8246,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.705333



WEIGHTED MODEL


,title,genres,similarity_score
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.930007
2264,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.851211
4780,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.786317
14813,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.770890
59767,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,0.730895
39485,Finding Dory (2016),Adventure|Animation|Comedy,0.718785
6258,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.712220
8246,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.706227
19870,Monsters University (2013),Adventure|Animation|Comedy,0.699954
48035,Lou (2017),Animation,0.673460


In [48]:
# Check whether the exact movie title exists

print("Exact title exists:",
      "The Matrix (1999)" in movie_indices)

# Search for Matrix movies
movie_features[
    movie_features["title"].str.contains(
        "Matrix",
        case=False,
        na=False
    )
][["movieId", "title", "genres"]].head(20)

Exact title exists: False


,movieId,title,genres
2480,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
6247,6365,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX
6809,6934,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX
9284,27660,"Animatrix, The (2003)",Action|Animation|Drama|Sci-Fi
28793,132490,Return to Source: The Philosophy of The Matrix...,Documentary
39665,157721,Armitage: Dual Matrix (2002),Action|Adventure|Animation|Sci-Fi|Thriller
46345,172255,The Matrix Revisited (2001),Documentary
49723,179489,The Living Matrix (2009),Documentary
50471,181103,Matrix of Evil (2003),Documentary


In [49]:
def genre_overlap(movie_title, recommendations):
    target_genres = set(
        movie_features.loc[
            movie_features["title"] == movie_title,
            "genres"
        ].iloc[0].split("|")
    )

    overlaps = []

    for genres in recommendations["genres"]:
        recommended_genres = set(genres.split("|"))
        overlaps.append(
            len(target_genres.intersection(recommended_genres)) > 0
        )

    return sum(overlaps)


print(
    "Toy Story original genre overlap:",
    genre_overlap("Toy Story (1995)", original_recs),
    "/ 10"
)

print(
    "Toy Story weighted genre overlap:",
    genre_overlap("Toy Story (1995)", weighted_recs),
    "/ 10"
)

Toy Story original genre overlap: 10 / 10
Toy Story weighted genre overlap: 10 / 10


In [50]:
matrix_titles = movie_features[
    movie_features["title"].str.contains(
        "Matrix",
        case=False,
        na=False
    )
]["title"].tolist()

matrix_titles[:20]

['Matrix, The (1999)',
 'Matrix Reloaded, The (2003)',
 'Matrix Revolutions, The (2003)',
 'Animatrix, The (2003)',
 'Return to Source: The Philosophy of The Matrix (2004)',
 'Armitage: Dual Matrix (2002)',
 'The Matrix Revisited (2001)',
 'The Living Matrix (2009)',
 'Matrix of Evil (2003)']

In [51]:
matrix_titles = movie_features[
    movie_features["title"].str.contains(
        "Matrix",
        case=False,
        na=False
    )
]["title"].tolist()

matrix_titles[:20]

['Matrix, The (1999)',
 'Matrix Reloaded, The (2003)',
 'Matrix Revolutions, The (2003)',
 'Animatrix, The (2003)',
 'Return to Source: The Philosophy of The Matrix (2004)',
 'Armitage: Dual Matrix (2002)',
 'The Matrix Revisited (2001)',
 'The Living Matrix (2009)',
 'Matrix of Evil (2003)']

In [53]:
def genre_overlap(movie_title, recommendations):
    target_genres = set(
        movie_features.loc[
            movie_features["title"] == movie_title,
            "genres"
        ].iloc[0].split("|")
    )

    overlaps = []

    for genres in recommendations["genres"]:
        recommended_genres = set(genres.split("|"))

        overlaps.append(
            len(target_genres.intersection(recommended_genres)) > 0
        )

    return sum(overlaps)


# Toy Story
toy_original = recommend_movies("Toy Story (1995)", n=10)
toy_weighted = recommend_movies_weighted("Toy Story (1995)", n=10)

print(
    "Toy Story - Original:",
    genre_overlap("Toy Story (1995)", toy_original),
    "/ 10"
)

print(
    "Toy Story - Weighted:",
    genre_overlap("Toy Story (1995)", toy_weighted),
    "/ 10"
)


# Find an exact Matrix title from the dataset
matrix_titles = movie_features[
    movie_features["title"].str.contains(
        "Matrix",
        case=False,
        na=False
    )
]["title"].tolist()

matrix_title = matrix_titles[0]

matrix_original = recommend_movies(matrix_title, n=10)
matrix_weighted = recommend_movies_weighted(matrix_title, n=10)

print(
    f"{matrix_title} - Original:",
    genre_overlap(matrix_title, matrix_original),
    "/ 10"
)

print(
    f"{matrix_title} - Weighted:",
    genre_overlap(matrix_title, matrix_weighted),
    "/ 10"
)

Toy Story - Original: 10 / 10
Toy Story - Weighted: 10 / 10
Matrix, The (1999) - Original: 10 / 10
Matrix, The (1999) - Weighted: 10 / 10


In [54]:
joblib.dump(
    weighted_tfidf,
    "../models/weighted_tfidf_vectorizer.pkl"
)

print("Weighted TF-IDF vectorizer saved successfully.")

Weighted TF-IDF vectorizer saved successfully.


In [55]:
import os

print(
    os.path.exists(
        "../models/weighted_tfidf_vectorizer.pkl"
    )
)

True


In [56]:
from scipy import sparse

sparse.save_npz(
    "../models/weighted_tfidf_matrix.npz",
    weighted_tfidf_matrix
)

print("Weighted TF-IDF matrix saved successfully.")

Weighted TF-IDF matrix saved successfully.


In [57]:
print(
    os.path.exists(
        "../models/weighted_tfidf_matrix.npz"
    )
)

True


In [58]:
import sys
import os

sys.path.append("../")

from src.content_based.vectorizer import (
    load_movie_features,
    create_tfidf_vectorizer
)

from src.content_based.similarity import (
    create_movie_index,
    recommend_movies
)

print("Modules imported successfully.")

Modules imported successfully.


In [59]:
movie_features_src = load_movie_features(
    "../data/processed/movie_features.csv"
)

print(movie_features_src.shape)
print(movie_features_src.columns.tolist())

(62423, 5)
['movieId', 'title', 'genres', 'tag', 'content']


In [60]:
movie_indices_src = create_movie_index(
    movie_features_src
)

print(
    movie_indices_src["Toy Story (1995)"]
)

0


In [61]:
test_recommendations = recommend_movies(
    "Toy Story (1995)",
    movie_features_src,
    weighted_tfidf_matrix,
    movie_indices_src,
    n=10
)

display(test_recommendations)

,movieId,title,genres,similarity_score
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.930007
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.851211
4780,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.786317
14813,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.770890
59767,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,0.730895
39485,157296,Finding Dory (2016),Adventure|Animation|Comedy,0.718785
6258,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.712220
8246,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.706227
19870,103141,Monsters University (2013),Adventure|Animation|Comedy,0.699954
48035,175831,Lou (2017),Animation,0.673460


In [1]:
import sys
sys.path.append("../")

from src.recommend import recommend

recommend("Toy Story (1995)", n=10)

,movieId,title,genres,similarity_score
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.930007
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.851211
4780,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.786317
14813,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.770890
59767,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,0.730895
39485,157296,Finding Dory (2016),Adventure|Animation|Comedy,0.718785
6258,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.712220
8246,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.706227
19870,103141,Monsters University (2013),Adventure|Animation|Comedy,0.699954
48035,175831,Lou (2017),Animation,0.673460
